In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

VARIANT = 'occupancy'   
#шлях до даних
DATA = Path('data')
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)

if VARIANT == 'bean':
    df = pd.read_csv(DATA / 'drybean.csv')
    target = 'Class'
    features = [c for c in df.columns if c not in ('Class', 'split')]
else:
    df = pd.read_csv(DATA / 'occupancy.csv')
    target = 'Occupancy'
    features = ['Temperature', 'Humidity', 'Light', 'CO2', 'HumidityRatio']

#дані вже розмічені
train, val, test = (df[df['split'] == s] for s in ('train', 'val', 'test'))
X_tr, y_tr = train[features], train[target]
X_va, y_va = val[features], val[target]
X_te, y_te = test[features], test[target]
print('train / validation / test:', len(train), len(val), len(test))

# 1.1
print("Features Q-ty: ", X_tr.shape[1])
print("Classes Q-ty: ", y_tr.nunique())

# 1.2
y_tr.value_counts().plot(kind='bar', rot=0)
plt.title('Class dissipation in \"train\"')
plt.xlabel('Occupancy (0 — none, 1 — people)')
plt.ylabel('Sample Q-ty')
plt.show()

# 1.3
majority = y_tr.value_counts().idxmax()
lazy_acc = (y_va == majority).mean()
print(f'Accuracy of lazy model: {lazy_acc:.4f}')

# 1.4
bal = y_tr.value_counts(normalize=True).values.round(2)
print("Balancing: ", bal[0], "/", bal[1])

#
print('---------------------------------------')
#

# 2.1
tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(X_tr, y_tr)

plt.figure(figsize=(20, 8))
plot_tree(tree, feature_names=features, class_names=["0", "1"], max_depth=3, filled=True, fontsize=10)
plt.show()

# 2.2

acc_tr = accuracy_score(y_tr, tree.predict(X_tr))
acc_va = accuracy_score(y_va, tree.predict(X_va))

print("Train: ", acc_tr, "Validation: ", acc_va)

# 2.3

print(export_text(tree, feature_names=features, decimals=3))

#
print('---------------------------------------')
#

# 3
forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X_tr, y_tr)

# 3.1
plt.figure(figsize=(25, 12))
plot_tree(forest.estimators_[0], feature_names=features, class_names=["0", "1"], filled=True, max_depth=3, fontsize=8)
plt.show()

# 3.2
def metrics(model):
    p = model.predict(X_va)
    return {
        'accuracy':  accuracy_score(y_va, p),
        'precision': precision_score(y_va, p, average='macro'),
        'recall':    recall_score(y_va, p, average='macro'),
    }

metric = pd.DataFrame({'Tree': metrics(tree), 'Forest': metrics(forest)}).T
print(metric.round(4))

# 3.3
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, m) in zip(axes, [('Tree (depth=3)', tree), ('Forest (100)', forest)]):
    ConfusionMatrixDisplay.from_predictions(y_va, m.predict(X_va), ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

diff = metric.loc['Forest', 'accuracy'] - metric.loc['Tree', 'accuracy']
print(f'Forest − Tree: {diff*100:+.2f} п.п.')
for name, m in [('Tree', tree), ('Forest', forest)]:
    cm = confusion_matrix(y_va, m.predict(X_va))
    print(f'{name}: пропущені = {cm[1,0]}, хибні тривоги = {cm[0,1]}')

# 4.1
print("-------------------------------------")

tree_acc_exp_tr = []
tree_acc_exp_va = []
depths = range(1,21)
for d in depths:
    tree_exp_tr = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)

    tree_acc_exp_tr.append((accuracy_score(y_tr, tree_exp_tr.predict(X_tr))))
    tree_acc_exp_va.append((accuracy_score(y_va, tree_exp_tr.predict(X_va))))

plt.figure(figsize=(8, 4.5))
plt.plot(depths, tree_acc_exp_tr, 'o-', label="train")
plt.plot(depths, tree_acc_exp_va, 's-', label="validation")
plt.xticks(depths)
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.title('Max Depth -> Accuracy')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 4.2
forest_acc_exp_tr = []
forest_acc_exp_va = []
test = [1, 5, 10, 25, 50, 100, 200, 300]
for n in test:
    forest_exp_tr = RandomForestClassifier(n_estimators=n, random_state=0).fit(X_tr, y_tr)

    forest_acc_exp_tr.append(accuracy_score(y_tr, forest_exp_tr.predict(X_tr)))
    forest_acc_exp_va.append(accuracy_score(y_va, forest_exp_tr.predict(X_va)))

plt.figure(figsize=(8, 4.5))
plt.plot(test, forest_acc_exp_tr, 'o-', label="train")
plt.plot(test, forest_acc_exp_va, 's-', label="validation")
plt.xticks(test, test)
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.title('N Estimators -> Accuracy')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 4(в) Найкраща глибина — тільки за validation
best_i = int(np.argmax(tree_acc_exp_va))
best_depth = list(depths)[best_i]

best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=0).fit(X_tr, y_tr)
best_acc_va = accuracy_score(y_va, best_tree.predict(X_va))
print(f'Найкраща глибина: {best_depth}, accuracy validation = {best_acc_va:.4f}')

# Test — ОДИН раз, для дерева і лісу
test_tree = accuracy_score(y_te, best_tree.predict(X_te))
test_forest = accuracy_score(y_te, forest.predict(X_te))
print(f'TEST дерево (depth={best_depth}): {test_tree:.4f}')
print(f'TEST ліс (100 дерев):           {test_forest:.4f}')

# 5.1
imp = pd.DataFrame({
    'Tree (depth=3)': tree.feature_importances_,
    'Forest (100)':   forest.feature_importances_,
}, index=features)

imp.sort_values('Forest (100)').plot(kind='barh', figsize=(8, 4.5))
plt.xlabel('feature_importances_')
plt.title('Feature importance')
plt.grid(alpha=0.3, axis='x')
plt.show()

# 5.2
top3 = imp['Forest (100)'].sort_values(ascending=False).head(3)
print(top3.round(4))

# 5.3
feats_noL = [f for f in features if f != 'Light']
forest_noL = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr[feats_noL], y_tr)

cmp = pd.DataFrame({
    'validation': [accuracy_score(y_va, forest.predict(X_va)),
                   accuracy_score(y_va, forest_noL.predict(X_va[feats_noL]))],
    'test':       [accuracy_score(y_te, forest.predict(X_te)),
                   accuracy_score(y_te, forest_noL.predict(X_te[feats_noL]))],
}, index=['Forest (all features)', 'Forest (no Light)'])
print(cmp.round(4))

# 5.4
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, m, cols) in zip(axes, [('All features', forest, features),
                                      ('No Light', forest_noL, feats_noL)]):
    ConfusionMatrixDisplay.from_predictions(y_te, m.predict(X_te[cols]), ax=ax, colorbar=False)
    ax.set_title(f'{name}, test')
plt.tight_layout()
plt.show()


# 6.1
typical = X_tr.median()          # «типова» кімната з train

cases = {
    'темно, свіже повітря':      {'Light': 0,   'CO2': 450},
    'світло, багато CO2':        {'Light': 450, 'CO2': 1000},
    'темно, але багато CO2':     {'Light': 0,   'CO2': 1100},
    'світло, але свіже повітря': {'Light': 450, 'CO2': 450},
    'напівтемрява, CO2 помірний':{'Light': 300, 'CO2': 750},
}

rows = []
for name, change in cases.items():
    r = typical.copy()
    for k, v in change.items():
        r[k] = v
    rows.append(r)

X_new = pd.DataFrame(rows, columns=features, index=cases.keys())
X_new

# 6.2 and 6.3
proba = forest.predict_proba(X_new)

table = pd.DataFrame({
    'дерево (best)': best_tree.predict(X_new),
    'ліс':           forest.predict(X_new),
    'P(0) ліс':      proba[:, 0].round(2),
    'P(1) ліс':      proba[:, 1].round(2),
}, index=X_new.index)
table['різні?'] = np.where(table['дерево (best)'] != table['ліс'], 'так', '')
table